# SoccerNet-GSR (Game State Reconstruction) Kaggle GPU Baseline

This notebook executes the official SoccerNet-GSR (sn-gamestate) perception baseline on a single validation sequence, validates tracking quality, and converts raw predictions into the project's Canonical Tracking Schema (v0.2.0).

**Hardware requirement:** Kaggle GPU (T4 x 2 or P100) or Google Colab GPU.

## Stage 1: GPU & CUDA Hardware Verification

In [ ]:
!nvidia-smi
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    x = torch.randn((1024, 1024), device="cuda")
    print("CUDA memory allocated:", torch.cuda.memory_allocated(0), "bytes")
else:
    raise RuntimeError("No GPU detected. Switch runtime accelerator to GPU.")

## Stage 2: Repository Clone & Workspace Setup

In [ ]:
!git clone https://github.com/manassdhumal/ball-free-game-state-reconstruction.git capstone
%cd capstone
!git checkout gpu-gsr-kaggle
!git clone https://github.com/SoccerNet/sn-gamestate.git soccernet-gamestate

## Stage 3: Dependency Installation (TrackLab, MMCV, MMDetection, MMOCR)

In [ ]:
!python kaggle/setup_gsr.py --gsr-dir soccernet-gamestate --output-dir results/gsr_kaggle

## Stage 4: Dataset Mount & Checkpoints Setup
Ensure the single validation sequence (e.g., ) is placed in .

In [ ]:
# Verify sequence data presence
from pathlib import Path
seq_dir = Path("data/SoccerNetGS/valid/SNGS-04")
if seq_dir.exists():
    print(f"Sequence found with {len(list(seq_dir.iterdir()))} files.")
else:
    print(f"Sequence {seq_dir} not yet downloaded. Follow docs/gsr_access_checklist.md.")

## Stage 5: Bounded One-Sequence Baseline Inference

In [ ]:
!python kaggle/run_gsr.py --sequence-id SNGS-04 --split valid --output-dir results/gsr_kaggle

## Stage 6: Canonical Tracking Export & Quality Summary

In [ ]:
!python kaggle/export_gsr.py --raw-output results/gsr_kaggle/predictions/SNGS-04.csv --output-dir results/gsr_kaggle --match-id gsr_valid_sngs04

## Stage 7: Downstream Local Integration Test

In [ ]:
from src.io.gsr_adapter import load_canonical_gsr_tracking
canonical_csv = Path("results/gsr_kaggle/exports/gsr_tracking_canonical.csv")
if canonical_csv.exists():
    df = load_canonical_gsr_tracking(canonical_csv)
    print("Successfully verified canonical tracking with", len(df), "rows.")
    print(df.head())
else:
    print("Canonical export file pending inference output.")